# Practice 2.4 · 长度控制精度

> **对应知识点**：EP2 · KP4 · 控制输出长度
> **测什么**：3 种长度约束方式的实际精度
> **预计时间**：5 分钟

---

## 你会看到什么

同一话题（"健康饮食"），三种长度约束：

- 版本 A · 具体字数（"150 字"）
- 版本 B · Hard cap（"不超过 30 字"）
- 版本 C · 上限暗示词（"尽可能详细，超越基础"）

**用 assert 硬测精度**。


In [ ]:
# ============ 前置：安装 SDK + 配置 client ============
!pip install openai -q

from openai import OpenAI
from google.colab import userdata   # Colab；Modelscope 上用 os.environ 或直接填 key

# 推荐用 Secrets 存 key（不要硬编码）
API_KEY = userdata.get("DEEPSEEK_KEY")   # Colab Secrets
# 或 Modelscope 上：
# API_KEY = "sk-你的DeepSeek key"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com/v1"
)

MODEL = "deepseek-chat"

def call(prompt, **kwargs):
    """简化的 chat 调用。默认 temperature=0 保证实验可重现。"""
    kwargs.setdefault("temperature", 0)
    kwargs.setdefault("max_tokens", 512)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return r.choices[0].message.content


## 3 版 Prompt

In [ ]:
TOPIC = "健康饮食"

# 版本 A · 具体字数
PROMPT_A = f"用 150 字介绍{TOPIC}。"

# 版本 B · Hard cap
PROMPT_B = f"用不超过 30 字介绍{TOPIC}。一句话。"

# 版本 C · 上限暗示
PROMPT_C = f"介绍{TOPIC}。尽可能详细，超越基础介绍，做一个全面的入门指南。"

# 版本 D · 已塌的形容词（对照组）
PROMPT_D = f"详细介绍{TOPIC}。要深入一点，不要太浅。"

## 跑测试 + 长度验证

In [ ]:
def count_chars(text):
    return len(text.strip())

for label, prompt, expected in [
    ("A · 具体字数 (~150)", PROMPT_A, "120-200 字"),
    ("B · Hard cap (≤30)", PROMPT_B, "≤ 30 字"),
    ("C · 上限暗示", PROMPT_C, ">= 500 字"),
    ("D · 形容词（已塌）", PROMPT_D, "不确定"),
]:
    out = call(prompt, max_tokens=2000)
    length = count_chars(out)
    print(f"=== 版本 {label} ===")
    print(f"期望：{expected}")
    print(f"实际长度：{length} 字")
    print(f"输出前 80 字：{out[:80]}...")
    print()

# 观察 D 的长度：会发现"详细"这种形容词已经不再驱动长度扩展
# 而 C 的上限暗示词能真的把输出拉到饱和

## 硬 assert · 验证精度

In [ ]:
out_a = call(PROMPT_A, max_tokens=500)
out_b = call(PROMPT_B, max_tokens=200)
out_c = call(PROMPT_C, max_tokens=2000)

len_a, len_b, len_c = len(out_a), len(out_b), len(out_c)

print(f"A (期望 ~150): {len_a} 字")
print(f"B (期望 ≤30):  {len_b} 字")
print(f"C (期望长):    {len_c} 字")

assert len_b < len_c, f"Hard cap 应该 < 上限暗示。B={len_b}, C={len_c}"
assert len_c > 300, f"上限暗示应该拉出长内容。C={len_c}"
print("\n✅ 通过：3 种约束都按预期生效")

## 一句心法

> **要长就给数字，要短就给数字。形容词已经不 work 了**。
>
> "详细 / 深入 / 全面" 这类形容词在 Claude 4.5+ 上已塌 —— 模型不会因此变长。
> 真正拉长的是**上限暗示词**（"尽可能" / "超越基础" / "全面"）。
